In [ ]:
import numpy as np
import nd2
from skimage.io import imread, imsave
import pyclesperanto_prototype as cle

from pathlib import Path

import warnings

In [ ]:
# Input variables

gpu = False # Use GPU
sigma = 50 # Sigma of the Gaussian blur
raw_images = './raw/' # Folder containing the splitted raw images

input_path = Path(raw_images)
input_path = list(input_path.glob('*.nd2'))

output_list_path = [Path('./bf_images/'+img.stem+'.tif') for img in input_path]

In [ ]:
def normalize_background(img, sigma=sigma, gpu=gpu):
    
    """
    This function loads an image and performs the division of the input by a blurred filtered version of itself. Give back a 8-bit clipped version of the brightfield image.
    """
    intensity_normalized = None
    
    if gpu:
        raise Exception('GPU processing Not implemented') 
        # pushed = cle.push(img)
        # intensity_normalized = cle.divide_by_gaussian_background(pushed, result_substract, sigma, sigma, 0)
        # intensity_normalized = np.asarray(intensity_normalized) #img is pulled from GPU memory
    
    else:
        intensity_normalized = cle.divide_by_gaussian_background(img, intensity_normalized, sigma,sigma,0)
        intensity_normalized = np.asarray(intensity_normalized)
        max_value = np.max(intensity_normalized)
        min_value = np.min(intensity_normalized)
        intensity_normalized = (intensity_normalized - min_value)/(max_value-min_value)*255
        intensity_normalized = intensity_normalized.astype(np.uint8)
    
    return intensity_normalized

In [ ]:
warnings.filterwarnings(action='ignore')
for inp, output in zip(input_path, output_list_path):
    img = nd2.imread(inp.as_posix())[:,0,:,:]
    out_img = normalize_background(img, sigma=sigma, gpu=gpu)
    imsave(output.as_posix(),out_img)

warnings.filterwarnings(action="default")